# Hafta 4 — Genelleme ve Model Değerlendirme

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

Deneme sınavlarının cevaplarını ezberleyen bir öğrenci aynı soruları tam puanla çözer. Ancak gerçek sınavda yeni sorularla karşılaşınca zorlanır. Modeller de eğitim örneklerini bu şekilde ezberleyebilir; bu duruma **aşırı öğrenme** denir. Tersi de mümkündür: Çok basit kalan bir model, eğitim örneklerinde bile başarılı olamaz. Buna da **eksik öğrenme** adı verilir.

Doğru dengeyi bulmak için model, verinin farklı parçalarıyla defalarca sınanır. Ayrıca modelin gereğinden karmaşık hale gelmesini engelleyen sınırlamalar kullanılır.

---

## Bu Haftanın Hedefleri

- Aşırı ve eksik öğrenmeyi tanımak
- Yanlılık-varyans dengesini anlamak
- k-katlı çapraz doğrulamayı sıfırdan kodlamak
- Ridge ve Lasso düzenlileştirmesinin etkisini görmek

---

## 1. Aşırı ve Eksik Öğrenme

Asıl amaç, modelin **daha önce görmediği** verilerde iyi tahmin yapmasıdır. Buna **genelleme** denir.

| | Eksik öğrenme | Aşırı öğrenme |
|---|---|---|
| Belirti | Eğitim ve test hatası ikisi de yüksek | Eğitim hatası düşük, test hatası yüksek |
| Neden | Model çok basit | Model çok karmaşık veya veri az |
| Çözüm | Daha karmaşık model, daha iyi öznitelikler | Daha fazla veri, daha basit model, düzenlileştirme |

**Deney:** Farklı derecelerde polinom modeller eğitip eğitim ve test hatalarını karşılaştıralım.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(0)
x_tr = np.sort(rng.uniform(0, 4, 20)); y_tr = np.sin(1.5 * x_tr) + rng.normal(0, 0.2, 20)
x_te = np.sort(rng.uniform(0, 4, 200)); y_te = np.sin(1.5 * x_te) + rng.normal(0, 0.2, 200)

dereceler = range(1, 16)
egitim, test = [], []
for d in dereceler:
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(x_tr[:, None], y_tr)
    egitim.append(mean_squared_error(y_tr, m.predict(x_tr[:, None])))
    test.append(mean_squared_error(y_te, m.predict(x_te[:, None])))

plt.plot(dereceler, egitim, "o-", label="Eğitim hatası")
plt.plot(dereceler, test, "s-", label="Test hatası")
plt.yscale("log"); plt.xlabel("Polinom derecesi"); plt.legend(); plt.show()

Eğitim hatası karmaşıklık arttıkça **hep azalır**; test hatası ise önce azalır, sonra artar. Bu yüzden model, eğitim hatasına bakılarak seçilemez.

---

## 2. Yanlılık-Varyans Dengesi

Aynı modeli farklı eğitim setleriyle tekrar tekrar eğittiğimizi düşünelim:

- **Yanlılık (bias):** Modelin tahminlerinin gerçek değerden **sistematik** olarak sapması. Eğri bir ilişkiye düz çizgi uydurmak yüksek yanlılık yaratır.
- **Varyans:** Eğitim verisi değişince modelin tahminlerinin **ne kadar değiştiği**. Çok esnek bir model, verideki gürültüyü de öğrenir ve her seferinde farklı bir sonuç verir.

```
Test hatası = Yanlılık² + Varyans + Gürültü (hiçbir modelin azaltamayacağı kısım)
```

Basit modellerde yanlılık, karmaşık modellerde varyans yüksektir. Amaç, toplamı en küçük yapan karmaşıklığı bulmaktır.

---

## 3. k-Katlı Çapraz Doğrulama

Veriyi bir kez eğitim/doğrulama olarak bölmek şansa bağlıdır. Bu yüzden şöyle yapılır:

1. Veriyi k eşit parçaya (kat) böl.
2. Her turda bir katı doğrulama, kalanları eğitim için kullan.
3. k skorun ortalamasını al.

```
k = 5:   Tur 1: [DOĞR][eğit][eğit][eğit][eğit]
         Tur 2: [eğit][DOĞR][eğit][eğit][eğit]
         ...
         Tur 5: [eğit][eğit][eğit][eğit][DOĞR]   →  sonuç = 5 skorun ortalaması
```

Her örnek tam bir kez doğrulamada kullanılır.

In [ ]:
from sklearn.base import clone

def k_katli_cv(model, X, y, k=5, seed=42):
    katlar = np.array_split(np.random.default_rng(seed).permutation(len(X)), k)
    skorlar = []
    for i in range(k):
        dogr = katlar[i]
        egit = np.concatenate([katlar[j] for j in range(k) if j != i])
        m = clone(model).fit(X[egit], y[egit])
        skorlar.append(mean_squared_error(y[dogr], m.predict(X[dogr])))
    return np.array(skorlar)

for d in [1, 3, 5, 10]:
    s = k_katli_cv(make_pipeline(PolynomialFeatures(d), LinearRegression()), x_tr[:, None], y_tr)
    print(f"Derece {d:2d}: CV hatası = {s.mean():.4f} ± {s.std():.4f}")

scikit-learn karşılığı: `cross_val_score(model, X, y, cv=5)`. Sınıflandırmada sınıf oranlarını koruyan `StratifiedKFold`, zaman serilerinde ise `TimeSeriesSplit` kullanılır.

---

## 4. Düzenlileştirme: Ridge ve Lasso

Aşırı öğrenen modellerin ağırlıkları genellikle çok büyüktür. Düzenlileştirme, kayıp fonksiyonuna **büyük ağırlıklar için bir ceza** ekler:

```
Ridge (L2):  Kayıp = MSE + λ · Σ w²
Lasso (L1):  Kayıp = MSE + λ · Σ |w|
```

- λ = 0 ise sıradan regresyon elde edilir; λ çok büyükse tüm ağırlıklar sıfıra yaklaşır.
- **Ridge** ağırlıkları küçültür ama sıfır yapmaz.
- **Lasso** önemsiz özniteliklerin ağırlıklarını **tam sıfıra** indirir; yani öznitelik seçimi de yapar.
- Ceza bütün ağırlıklara eşit uygulandığı için öznitelikler önce **ölçeklenmelidir**.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge, Lasso

X, y = load_diabetes(return_X_y=True)
X = StandardScaler().fit_transform(X)
lambdalar = np.logspace(-2, 3, 50)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(lambdalar, [Ridge(alpha=l).fit(X, y).coef_ for l in lambdalar]); ax[0].set_title("Ridge")
ax[1].plot(lambdalar / 10, [Lasso(alpha=l / 10, max_iter=10000).fit(X, y).coef_ for l in lambdalar]); ax[1].set_title("Lasso")
for a in ax: a.set_xscale("log"); a.set_xlabel("λ"); a.set_ylabel("katsayı")
plt.show()

Grafikte Ridge katsayılarının yavaşça sıfıra yaklaştığını, Lasso katsayılarının ise tek tek **tam sıfıra düştüğünü** göreceksiniz. En iyi λ çapraz doğrulama ile seçilir (`RidgeCV`, `LassoCV`).

---

## 5. Öğrenme Eğrisi

Öğrenme eğrisi, eğitim verisi arttıkça eğitim ve doğrulama hatasının nasıl değiştiğini gösterir:

- **İki eğri yakın ama ikisi de yüksek** → eksik öğrenme. Daha fazla veri işe yaramaz; daha güçlü bir model gerekir.
- **Aralarında büyük boşluk var** → aşırı öğrenme. Daha fazla veri veya düzenlileştirme işe yarar.

In [ ]:
from sklearn.model_selection import learning_curve

model = make_pipeline(PolynomialFeatures(10), StandardScaler(), LinearRegression())
boyut, egt, dgr = learning_curve(model, x_te[:, None], y_te, cv=5, scoring="neg_mean_squared_error",
                                 train_sizes=np.linspace(0.1, 1.0, 8))
plt.plot(boyut, -egt.mean(axis=1), "o-", label="Eğitim")
plt.plot(boyut, -dgr.mean(axis=1), "s-", label="Doğrulama")
plt.xlabel("Eğitim örneği sayısı"); plt.ylabel("MSE"); plt.legend(); plt.show()

---

## 6. Laboratuvar

1. Derece 15 polinomu Ridge ile λ = 1e-8, 1e-2, 1 ve 100 değerleriyle eğitip eğrileri aynı grafikte çizin. Hangi λ aşırı öğrenmeyi düzeltiyor, hangisi eksik öğrenmeye yol açıyor?
2. Kendi `k_katli_cv` fonksiyonunuzla 20 farklı λ için CV hatasını hesaplayın ve en iyi λ'yı bulun.
3. Diabetes verisinde λ arttıkça Lasso'nun kaç özniteliği sıfırladığını tablo halinde gösterin.

---

## 7. Özet

- Amaç **genellemedir**; eğitim hatası tek başına yanıltıcıdır.
- Eksik öğrenme = yüksek yanlılık, aşırı öğrenme = yüksek varyans.
- k-katlı çapraz doğrulama daha güvenilir bir performans tahmini verir.
- Ridge ağırlıkları küçültür, Lasso bazılarını sıfırlar; λ çapraz doğrulamayla seçilir.

## Kendinizi Sınayın

1. Eğitim doğruluğu %99, test doğruluğu %70 olan bir model için iki çözüm önerin.
2. 10-katlı çapraz doğrulamada model kaç kez eğitilir?
3. 1000 öznitelikten yalnızca 10'unun önemli olduğunu düşünüyorsanız Ridge mi, Lasso mu seçersiniz?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.